**Cell 1：環境設定與掛載 Google Drive**


這段會安裝並載入需要的套件，以及設定你的 Claude API 金鑰。

In [1]:
# 1. 安裝 Claude 的官方套件
!pip install anthropic

import json
import time
import glob
import anthropic
from google.colab import drive

# 2. 掛載 Google Drive (會跳出授權視窗)
drive.mount('/content/drive')

# 3. 初始化 Claude 客戶端 (⚠️ 請替換成你自己的 API Key)
client = anthropic.Anthropic(api_key="YOUR_API_KEY_HERE")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 19.6 MB/s eta 0:00:00
Mounted at /content/drive


**Cell 2：定義大腦評分邏輯 (System Prompt 與 API 函數)**


我將官方解答 answer 的文字也一併餵給 Claude，這會讓它在判斷 0.5 分和 0.2 分時變得極度精準！

In [5]:
# 這是我們精心設計的閱卷老師 Prompt
SYSTEM_PROMPT = """你是一個客觀且具備高度邏輯的考試評分專家。
我會給你一個「問題」、四個「選項 (A, B, C, D)」，以及官方的「正確解答內容」。
你的任務是判斷其他錯誤選項與正確解答之間的「語意相似度」、「部分正確性」或「數值接近程度」，並給予 0.0 到 1.0 的權重分數。

【評分標準】
- 1.0 分：完全正確（官方指定的正確解答必須是 1.0 分）。
- 0.5 分：大方向正確但細節偏差。若是「數值/數量型問題」，數值非常接近正確答案也可給此分。
- 0.2 分：提到相關物品或動作但邏輯錯誤。若是「數值/數量型問題」，數值有一定差距但非完全離譜。
- 0.0 分：完全瞎扯、毫無關聯、給出相反資訊，或是數值差距極大。

【輸出限制】
請「嚴格」只輸出一個 JSON 格式的字典，絕對不要輸出任何 Markdown 標記 (如 ```json)、也不要有任何其他解釋文字。
輸出範例：{"A": 1.0, "B": 0.5, "C": 0.0, "D": 0.2}
"""

def get_option_scores_from_llm(question, options_dict, gt_label, gt_text):
    """將題目與選項送給 Claude，取得權重 JSON"""
    user_content = f"""
    問題：{question}
    選項：
    A: {options_dict.get('A', '')}
    B: {options_dict.get('B', '')}
    C: {options_dict.get('C', '')}
    D: {options_dict.get('D', '')}
    正確解答是：{gt_label} ({gt_text}) (這個選項必須是 1.0 分)
    """

    try:
        # 推薦使用 haiku，速度最快且最便宜，做這種邏輯判斷非常適合
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=100,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_content}],
            temperature=0.1 # 溫度調低，確保 JSON 輸出穩定不亂加廢話
        )

        result_text = response.content[0].text.strip()
        # 清理可能存在的 Markdown 標籤，確保能轉成 Dictionary
        result_text = result_text.replace("```json", "").replace("```", "").strip()
        return json.loads(result_text)

    except Exception as e:
        print(f"  ❌ API 或解析錯誤: {e}")
        return None

**Cell 3：批次執行 9 份考卷！(核心引擎)**


這段程式會自動把 gt 的 0, 1, 2, 3 轉換成 A, B, C, D，然後逐題送給 Claude 算分。
(備註：請確認 dataset_dir 的路徑是否正確對應到你的資料夾)

In [6]:
# 📁 設定存放這 9 份 JSON 的資料夾路徑
# 假設你的資料夾結構是這樣，請根據實際情況修改
dataset_dir = '/content/drive/MyDrive/OVO-Bench/task_lists/'
file_list = glob.glob(f'{dataset_dir}*_questions.json')

# 用來存放所有幾百題的權重分數的終極字典
question_weights = {}

print(f"🚀 找到 {len(file_list)} 份考卷檔案，準備開始批次處理...")

# 用於將 0, 1, 2, 3 轉換為 A, B, C, D
idx_to_letter = {0: "A", 1: "B", 2: "C", 3: "D"}

for file_path in file_list:
    task_name = file_path.split('/')[-1].split('_')[0] # 擷取任務名稱，例如 STU, OCR
    print(f"\n📂 開始處理考卷：{task_name}")

    with open(file_path, 'r', encoding='utf-8') as f:
        raw_dataset = json.load(f)

    for idx, item in enumerate(raw_dataset):
        # 取得題號 (如果你的隨機森林系統是認數字，就直接用 item['id'])
        item_id = str(item['id'])
        question = item['question']
        opts_list = item['options']
        gt_idx = item['gt']
        gt_text = item['answer']

        # 將陣列選項轉成 ABCD 字典
        options = {
            "A": opts_list[0] if len(opts_list) > 0 else "",
            "B": opts_list[1] if len(opts_list) > 1 else "",
            "C": opts_list[2] if len(opts_list) > 2 else "",
            "D": opts_list[3] if len(opts_list) > 3 else ""
        }

        gt_label = idx_to_letter.get(gt_idx, "A")

        print(f"  [{idx+1}/{len(raw_dataset)}] 正在評分題號 {item_id} ... ", end="")

        # 包含失敗重試機制 (防禦 529 過載錯誤)
        max_retries = 3
        for attempt in range(max_retries):
            weights = get_option_scores_from_llm(question, options, gt_label, gt_text)
            if weights is not None:
                question_weights[item_id] = weights
                print(f"✅ {weights}")
                break
            else:
                print("⏳ 等待 5 秒後重試...")
                time.sleep(5)

        # 避免打 API 太快被限流，稍作延遲
        time.sleep(0.5)

🚀 找到 9 份考卷檔案，準備開始批次處理...

📂 開始處理考卷：ACR
  [1/109] 正在評分題號 1210 ... ✅ {'A': 0.2, 'B': 0.0, 'C': 0.2, 'D': 1.0}
  [2/109] 正在評分題號 1211 ... ✅ {'A': 0.0, 'B': 0.0, 'C': 0.0, 'D': 1.0}
  [3/109] 正在評分題號 1212 ... ✅ {'A': 0.0, 'B': 0.0, 'C': 0.0, 'D': 1.0}
  [4/109] 正在評分題號 1213 ... ✅ {'A': 1.0, 'B': 0.5, 'C': 0.2, 'D': 0.0}
  [5/109] 正在評分題號 1214 ... ✅ {'A': 1.0, 'B': 0.0, 'C': 0.2, 'D': 0.0}
  [6/109] 正在評分題號 1215 ... ✅ {'A': 1.0, 'B': 0.2, 'C': 0.2, 'D': 0.0}
  [7/109] 正在評分題號 1216 ... ✅ {'A': 1.0, 'B': 0.0, 'C': 0.0, 'D': 0.0}
  [8/109] 正在評分題號 1217 ... ✅ {'A': 0.2, 'B': 0.2, 'C': 1.0, 'D': 0.0}
  [9/109] 正在評分題號 1218 ... ✅ {'A': 1.0, 'B': 0.2, 'C': 0.0, 'D': 0.2}
  [10/109] 正在評分題號 1219 ... ✅ {'A': 0.2, 'B': 0.0, 'C': 0.0, 'D': 1.0}
  [11/109] 正在評分題號 1220 ... ✅ {'A': 1.0, 'B': 0.2, 'C': 0.2, 'D': 0.0}
  [12/109] 正在評分題號 1221 ... ✅ {'A': 0.2, 'B': 0.2, 'C': 1.0, 'D': 0.2}
  [13/109] 正在評分題號 1222 ... ✅ {'A': 0.2, 'B': 0.0, 'C': 0.2, 'D': 1.0}
  [14/109] 正在評分題號 1223 ... ✅ {'A': 0.0, 'B': 0.0, 'C': 0.0, 

**Cell 4：儲存終極成績對照表**

In [7]:
# 將所有算好的權重存檔到 Google Drive 的外層
output_path = '/content/drive/MyDrive/OVO-Bench/question_weights.json'

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(question_weights, f, indent=4, ensure_ascii=False)

print(f"\n🎉 大功告成！權重表已成功儲存至：{output_path}")

# 印出前 3 筆資料讓你安心檢查
print("\n🔍 預覽前 3 筆生成的權重字典：")
sample_keys = list(question_weights.keys())[:3]
for k in sample_keys:
    print(f"題號 {k}: {question_weights[k]}")


🎉 大功告成！權重表已成功儲存至：/content/drive/MyDrive/OVO-Bench/question_weights.json

🔍 預覽前 3 筆生成的權重字典：
題號 1210: {'A': 0.2, 'B': 0.0, 'C': 0.2, 'D': 1.0}
題號 1211: {'A': 0.0, 'B': 0.0, 'C': 0.0, 'D': 1.0}
題號 1212: {'A': 0.0, 'B': 0.0, 'C': 0.0, 'D': 1.0}
